# MedVision — External Validation on the Kermany/Guangzhou Pediatric Chest X-ray Cohort

**Purpose:** evaluate the frozen MedVision checkpoint on the complete available Kermany cohort without retraining or threshold tuning. The cohort contains `NORMAL` and `PNEUMONIA` images, so this experiment validates the model's **Pneumonia** output only; it does not claim validation of the other seven MedVision labels.

**Data handling:** the X-rays remain attached to the Kaggle Notebook as an input. Only small aggregate/result artifacts are written to `/kaggle/working`.

## Kaggle setup

Before running this notebook:

1. Create a Kaggle Notebook.
2. Add the dataset `paultimothymooney/chest-xray-pneumonia` through **Add Input**.
3. Add the MedVision trained checkpoint as another Kaggle Dataset/Input. The checkpoint should be the **DenseNet121 + Weighted BCE** checkpoint used by the MedVision API, normally named `densenet121_weighted_bce.pt`.
4. Enable a GPU accelerator if available.

Kaggle attaches datasets to the notebook environment, so the large image corpus does not need to be downloaded to your personal computer.

In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, shutil, subprocess

WORK = Path('/kaggle/working/medvision_external_validation')
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / 'outputs'
OUT.mkdir(exist_ok=True)

# Kaggle Input roots. We discover them rather than hard-coding dataset mount IDs.
INPUT_ROOT = Path('/kaggle/input')
print('Kaggle inputs:')
for p in sorted(INPUT_ROOT.iterdir()):
    print(' -', p)

# Locate the Kermany image directories.
kermany_candidates = []
for p in INPUT_ROOT.rglob('*'):
    if p.is_dir() and p.name in {'train', 'test', 'val', 'validation'}:
        kermany_candidates.append(p)

# Locate a checkpoint without assuming its dataset mount name.
checkpoint_candidates = list(INPUT_ROOT.rglob('*.pt')) + list(INPUT_ROOT.rglob('*.pth')) + list(INPUT_ROOT.rglob('*.ckpt'))
print('\nCheckpoint candidates:', checkpoint_candidates)
if not checkpoint_candidates:
    raise FileNotFoundError(
        'No MedVision checkpoint found under /kaggle/input. Add the trained .pt/.pth checkpoint as a Kaggle Dataset/Input.'
    )

CHECKPOINT = next((p for p in checkpoint_candidates if p.name == 'densenet121_weighted_bce.pt'), checkpoint_candidates[0])
print('Using checkpoint:', CHECKPOINT)

# The Kermany dataset has NORMAL/PNEUMONIA directories. Find the root containing both.
image_root = None
for p in [x for x in INPUT_ROOT.rglob('*') if x.is_dir()]:
    children = {c.name.upper() for c in p.iterdir() if c.is_dir()}
    if {'NORMAL', 'PNEUMONIA'}.issubset(children):
        image_root = p
        break

if image_root is None:
    raise FileNotFoundError('Could not find a directory containing both NORMAL and PNEUMONIA folders.')

print('Kermany image root:', image_root)


In [ ]:
# Clone the exact MedVision repository version used for the evaluation code.
REPO_DIR = WORK / 'MedVision'
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Rollins1989/MedVision.git', str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
print('MedVision source:', REPO_DIR)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve,
    log_loss, brier_score_loss
)

from src.models.factory import build_model
from src.preprocessing.transforms import build_transforms

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
# Load the frozen checkpoint exactly as the API does.
ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
required = {'model_name', 'label_names', 'num_labels', 'image_size', 'model_state_dict'}
missing = required - set(ckpt)
if missing:
    raise ValueError(f'Checkpoint missing required fields: {sorted(missing)}')

print('Checkpoint metadata:')
print(json.dumps({k: ckpt[k] for k in ['model_name','label_names','num_labels','image_size']}, indent=2, default=str))

model = build_model(ckpt['model_name'], num_labels=ckpt['num_labels'], pretrained=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
transform = build_transforms(ckpt['image_size'], train=False)

LABELS = list(ckpt['label_names'])
if 'Pneumonia' not in LABELS:
    raise ValueError(f'Checkpoint labels do not contain Pneumonia: {LABELS}')
PNEUMONIA_IDX = LABELS.index('Pneumonia')

print('Frozen model:', ckpt['model_name'])
print('Pneumonia index:', PNEUMONIA_IDX)
print('Device:', device)


In [ ]:
# Build the complete external cohort from every NORMAL and PNEUMONIA image.
records = []
for class_name, target in [('NORMAL', 0), ('PNEUMONIA', 1)]:
    folder = image_root / class_name
    for path in sorted(folder.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.jpeg', '.jpg', '.png', '.bmp'}:
            records.append({'path': str(path), 'image_id': path.name, 'external_label': target, 'class_name': class_name})

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError('No JPEG/PNG images found in NORMAL/PNEUMONIA.')

print('Complete cohort size:', len(df))
print(df['class_name'].value_counts().to_string())

# Quality audit before inference.
quality = []
for row in df.itertuples(index=False):
    try:
        with Image.open(row.path) as img:
            img.load()
            gray = np.asarray(img.convert('L'), dtype=np.float32)
            quality.append({
                'image_id': row.image_id,
                'class_name': row.class_name,
                'width': img.width,
                'height': img.height,
                'mode': img.mode,
                'pixel_std': float(gray.std()),
                'valid': bool(img.width >= 32 and img.height >= 32 and gray.std() >= 2.0),
            })
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        quality.append({'image_id': row.image_id, 'class_name': row.class_name, 'valid': False, 'error': str(exc)})

quality_df = pd.DataFrame(quality)
quality_df.to_csv(OUT / 'external_quality_audit.csv', index=False)
print('\nQuality audit:')
print(quality_df['valid'].value_counts(dropna=False).to_string())


In [ ]:
class ExternalChestXray(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row.path) as img:
            img = img.convert('L')
            tensor = self.transform(img)
        return tensor, float(row.external_label), row.image_id

valid_ids = set(quality_df.loc[quality_df['valid'] == True, 'image_id'])
valid_df = df[df['image_id'].isin(valid_ids)].reset_index(drop=True)
print(f'Valid images sent to model: {len(valid_df)} / {len(df)}')

loader = DataLoader(
    ExternalChestXray(valid_df, transform),
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

all_probs = []
all_y = []
all_ids = []
latencies = []

with torch.inference_mode():
    for xb, yb, ids in loader:
        xb = xb.to(device, non_blocking=True)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        logits = model(xb)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        latencies.extend([elapsed / len(xb) * 1000] * len(xb))
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
        all_y.append(yb.numpy())
        all_ids.extend(ids)

probs8 = np.concatenate(all_probs)
y = np.concatenate(all_y).astype(int)
pneumonia_probs = probs8[:, PNEUMONIA_IDX]

pred = (pneumonia_probs >= 0.5).astype(int)

predictions = pd.DataFrame({
    'image_id': all_ids,
    'y_true_pneumonia': y,
    'pneumonia_probability': pneumonia_probs,
    'prediction_threshold_0_5': pred,
    'latency_ms': latencies,
})
predictions.to_csv(OUT / 'external_predictions.csv', index=False)
np.save(OUT / 'external_pneumonia_probabilities.npy', pneumonia_probs)
np.save(OUT / 'external_pneumonia_labels.npy', y)

print('Inference complete:', len(predictions), 'images')


In [ ]:
# Primary external-validation metrics. Threshold-independent metrics are emphasized.
roc_auc = roc_auc_score(y, pneumonia_probs)
pr_auc = average_precision_score(y, pneumonia_probs)

cm = confusion_matrix(y, pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if tp + fn else float('nan')
specificity = tn / (tn + fp) if tn + fp else float('nan')
precision = tp / (tp + fp) if tp + fp else float('nan')
f1 = 2 * precision * sensitivity / (precision + sensitivity) if precision + sensitivity else 0.0
accuracy = (tp + tn) / cm.sum()

metrics = {
    'protocol': 'External validation of frozen MedVision checkpoint on complete Kermany/Guangzhou cohort available to the notebook',
    'dataset': 'Kermany pediatric chest X-ray dataset / Guangzhou Women and Children Medical Center',
    'n_images_source': int(len(df)),
    'n_images_evaluated': int(len(valid_df)),
    'n_images_excluded_quality': int(len(df) - len(valid_df)),
    'class_counts_source': {str(k): int(v) for k, v in df['class_name'].value_counts().to_dict().items()},
    'class_counts_evaluated': {str(k): int(v) for k, v in valid_df['class_name'].value_counts().to_dict().items()},
    'medvision_checkpoint': CHECKPOINT.name,
    'model_name': ckpt['model_name'],
    'pneumonia_label_index': int(PNEUMONIA_IDX),
    'threshold_for_binary_metrics': 0.5,
    'roc_auc': float(roc_auc),
    'pr_auc_average_precision': float(pr_auc),
    'accuracy_at_0_5': float(accuracy),
    'precision_at_0_5': float(precision),
    'recall_sensitivity_at_0_5': float(sensitivity),
    'specificity_at_0_5': float(specificity),
    'f1_at_0_5': float(f1),
    'confusion_matrix_tn_fp_fn_tp': [int(tn), int(fp), int(fn), int(tp)],
    'mean_latency_ms': float(np.mean(latencies)),
    'median_latency_ms': float(np.median(latencies)),
    'mean_pneumonia_probability': float(np.mean(pneumonia_probs)),
    'median_pneumonia_probability': float(np.median(pneumonia_probs)),
    'evaluation_notes': [
        'Checkpoint was loaded without retraining.',
        'No threshold was optimized on the external cohort.',
        'Only the Pneumonia output is evaluated because the external labels are binary Normal/Pneumonia.',
        'Normal is treated as the negative class for the binary Pneumonia endpoint.',
    ],
}

with open(OUT / 'external_validation.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))


In [ ]:
# Curves and a reliability diagram for the external pneumonia probability.
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y, pneumonia_probs)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('MedVision External Validation — Pneumonia ROC')
plt.legend()
plt.tight_layout()
plt.savefig(OUT / 'external_roc_curve.png', dpi=200)
plt.close()

prec, rec, _ = precision_recall_curve(y, pneumonia_probs)
plt.figure(figsize=(7, 6))
plt.plot(rec, prec, label=f'Average Precision = {pr_auc:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('MedVision External Validation — Pneumonia Precision-Recall')
plt.legend()
plt.tight_layout()
plt.savefig(OUT / 'external_pr_curve.png', dpi=200)
plt.close()

plt.figure(figsize=(6, 6))
plt.imshow(cm, interpolation='nearest')
plt.xticks([0, 1], ['Normal', 'Pneumonia'])
plt.yticks([0, 1], ['Normal', 'Pneumonia'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('External Confusion Matrix @ 0.5')
for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha='center', va='center')
plt.tight_layout()
plt.savefig(OUT / 'external_confusion_matrix.png', dpi=200)
plt.close()

# Reliability bins: no fitting/tuning; purely descriptive.
bins = np.linspace(0, 1, 11)
centers, observed, counts = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (pneumonia_probs >= lo) & (pneumonia_probs < hi if hi < 1 else pneumonia_probs <= hi)
    if mask.any():
        centers.append(float(pneumonia_probs[mask].mean()))
        observed.append(float(y[mask].mean()))
        counts.append(int(mask.sum()))

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], linestyle='--')
plt.plot(centers, observed, marker='o', label='External reliability')
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed pneumonia rate')
plt.title('External Pneumonia Reliability Diagram')
plt.legend()
plt.tight_layout()
plt.savefig(OUT / 'external_reliability.png', dpi=200)
plt.close()

print('Saved figures to:', OUT)


In [ ]:
# Package only small artifacts for transfer back to the MedVision repo.
manifest = {
    'source_dataset': 'paultimothymooney/chest-xray-pneumonia',
    'cohort': 'Kermany / Guangzhou Women and Children Medical Center',
    'images_used': int(len(valid_df)),
    'checkpoint': CHECKPOINT.name,
    'output_files': sorted(p.name for p in OUT.iterdir() if p.is_file()),
}
with open(OUT / 'MANIFEST.json', 'w') as f:
    json.dump(manifest, f, indent=2)

archive = shutil.make_archive(str(WORK / 'medvision_external_validation_results'), 'zip', OUT)
print('RESULTS ZIP:', archive)
print('Contents:')
for p in sorted(OUT.iterdir()):
    if p.is_file():
        print(f' - {p.name}: {p.stat().st_size / 1024:.1f} KB')


## What this experiment does *not* do

- It does **not** retrain MedVision on the external images.
- It does **not** optimize the 0.5 threshold using the external labels.
- It does **not** treat `NORMAL` as a negative label for the other seven MedVision diseases.
- It does **not** claim clinical validation.

The final ZIP contains only predictions, metrics, audit information, and figures—not the medical images.